In [2]:
import cv2
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

def parse_broken_info_fixed(xml_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        broken_start = None
        broken_end = None
        
        # track 요소들을 순회하면서 broken_start/end 찾기
        for track in root.findall('.//track'):
            label = track.get('label')
            
            if label == 'broken_start':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    broken_start = int(box.get('frame'))
            
            elif label == 'broken_end':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    broken_end = int(box.get('frame'))
        
        return broken_start, broken_end
    
    except Exception as e:
        print(f"❌ XML 파싱 오류 ({xml_file}): {e}")
        return None, None

def extract_broken_frames_fixed(video_dir, xml_dir, output_dir):
    """수정된 broken 구간 프레임 추출"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 비디오 파일 목록
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    
    print(f"총 {len(video_files)}개 비디오에서 broken 구간 추출 시작...")
    
    total_broken_frames = 0
    videos_with_broken = 0
    videos_without_broken = 0
    
    for video_file in tqdm(video_files, desc="비디오 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # 대응하는 XML 파일 찾기
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        if not os.path.exists(xml_path):
            print(f"⚠️ XML 파일 없음: {xml_file}")
            continue
        
        # broken 정보 추출 (수정된 함수)
        broken_start, broken_end = parse_broken_info_fixed(xml_path)
        
        if broken_start is None or broken_end is None:
            videos_without_broken += 1
            continue
        
        # 비디오 열기
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"❌ 비디오 열기 실패: {video_file}")
            continue
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"\n📹 {video_file}")
        print(f"   broken 구간: {broken_start} ~ {broken_end}")
        
        # broken 구간 유효성 검사
        if broken_end >= total_frames:
            print(f"   ⚠️ broken_end({broken_end})가 총 프레임({total_frames})보다 큼")
            broken_end = total_frames - 1
        
        frame_count = 0
        saved_frames = 0
        
        # 프레임별로 읽기
        while frame_count < total_frames:
            ret, frame = cap.read()
            
            if not ret:
                break
            
            # broken 구간에 있는 프레임만 저장
            if broken_start <= frame_count <= broken_end:
                # 파일명: 비디오이름_프레임번호_broken.jpg
                output_filename = f"{video_name}_{frame_count:03d}_broken.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                # JPG로 저장
                if cv2.imwrite(output_path, frame):
                    saved_frames += 1
            
            frame_count += 1
        
        cap.release()
        
        if saved_frames > 0:
            videos_with_broken += 1
            total_broken_frames += saved_frames
            print(f"   ✅ {saved_frames}개 broken 프레임 저장")
    
    print(f"\n🎉 추출 완료!")
    print(f"   broken 있는 비디오: {videos_with_broken}개")
    print(f"   broken 없는 비디오: {videos_without_broken}개") 
    print(f"   총 broken 프레임: {total_broken_frames}개")
    
    return total_broken_frames

def extract_normal_frames_fixed(video_dir, xml_dir, output_dir, sampling_rate=10):
    """
    정상 구간(broken가 아닌 구간) 프레임을 샘플링해서 저장 (수정된 버전)
    sampling_rate: N프레임마다 1개씩 저장 (기본값: 10프레임마다 1개)
    파일명: 비디오이름_프레임번호_normal.jpg
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    total_normal_frames = 0
    
    print(f"총 {len(video_files)}개 비디오에서 정상 구간 추출 시작... ({sampling_rate}프레임당 1개)")
    
    for video_file in tqdm(video_files, desc="정상 구간 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # XML에서 broken 구간 정보
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        # 수정된 파싱 함수 사용
        broken_start,broken_end = parse_broken_info_fixed(xml_path) if os.path.exists(xml_path) else (None, None)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
            
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_count = 0
        saved_normal = 0
        
        while frame_count < total_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 정상 구간 판단 (broken 구간이 아닌 곳)
            is_normal = True
            if broken_start is not None and broken_end is not None:
                if broken_start <= frame_count <= broken_end:
                    is_normal = False
            
            # 정상 구간이고 샘플링 조건에 맞으면 저장
            if is_normal and frame_count % sampling_rate == 0:
                output_filename = f"{video_name}_{frame_count:03d}_normal.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                if cv2.imwrite(output_path, frame):
                    saved_normal += 1
            
            frame_count += 1
        
        cap.release()
        total_normal_frames += saved_normal
        
        if saved_normal > 0 and broken_start is not None:
            print(f"   ✅ {video_file}: {saved_normal}개 정상 프레임")
    
    print(f"\n🎉 정상 구간 추출 완료!")
    print(f"   총 정상 프레임: {total_normal_frames}개")
    
    return total_normal_frames

# =============================================================================
# 완전한 실행 코드 (broken + normal)
# =============================================================================

# 경로 설정
video_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/videos/breaking_behavior/train/video"
xml_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/videos/breaking_behavior/train/label"
broken_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/frames/broken/train/images"
normal_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/frames/broken/normal/train/images"

print("🚀 절도 감지용 데이터셋 구성 시작!")
print("="*50)

# 1단계: broken 구간 프레임 추출 (수정된 함수)
print("\n📍 1단계: broken 구간 추출")
broken_frames = extract_broken_frames_fixed(video_dir, xml_dir, broken_output_dir)

# 2단계: 정상 구간 프레임 추출 (수정된 함수)
print("\n📍 2단계: 정상 구간 추출")
normal_frames = extract_normal_frames_fixed(video_dir, xml_dir, normal_output_dir, sampling_rate=10)

# 최종 결과
print("\n" + "="*50)
print("🎉 데이터셋 구성 완료!")
print(f"📊 최종 결과:")
print(f"   🔴 brokenn 프레임 : {broken_frames:,}개")
print(f"   🟢 Normal 프레임: {normal_frames:,}개")
print(f"   📈 총 프레임: {broken_frames + normal_frames:,}개")
if broken_frames > 0:
    print(f"   ⚖️ 비율 (normal:broken): {normal_frames // broken_frames}:1")
print("="*50)

🚀 절도 감지용 데이터셋 구성 시작!

📍 1단계: broken 구간 추출
총 642개 비디오에서 broken 구간 추출 시작...


비디오 처리:   0%|          | 0/642 [00:00<?, ?it/s]


📹 C_3_8_30_BU_SMA_09-27_10-49-40_CD_RGB_DF2_F3.mp4
   broken 구간: 75 ~ 150


비디오 처리:   0%|          | 1/642 [00:04<43:00,  4.03s/it]

   ✅ 76개 broken 프레임 저장

📹 C_3_8_32_BU_SMB_09-05_13-10-13_CC_RGB_DF2_M4.mp4
   broken 구간: 67 ~ 136


...

   ✅ C_3_8_15_BU_SMB_09-02_13-27-19_CB_RGB_DF2_M2.mp4: 9개 정상 프레임


정상 구간 처리: 100%|█████████▉| 641/642 [38:43<00:03,  3.52s/it]

   ✅ C_3_8_37_BU_DYB_10-16_14-24-01_CE_RGB_DF2_M1.mp4: 14개 정상 프레임


정상 구간 처리: 100%|██████████| 642/642 [38:47<00:00,  3.63s/it]

   ✅ C_3_8_4_BU_SYA_09-17_14-09-33_CC_RGB_DF2_F1.mp4: 11개 정상 프레임

🎉 정상 구간 추출 완료!
   총 정상 프레임: 6954개

🎉 데이터셋 구성 완료!
📊 최종 결과:
   🔴 brokenn 프레임 : 46,951개
   🟢 Normal 프레임: 6,954개
   📈 총 프레임: 53,905개
   ⚖️ 비율 (normal:broken): 0:1


In [3]:
import cv2
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

def parse_broken_info_fixed(xml_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        broken_start = None
        broken_end = None
        
        # track 요소들을 순회하면서 broken_start/end 찾기
        for track in root.findall('.//track'):
            label = track.get('label')
            
            if label == 'broken_start':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    broken_start = int(box.get('frame'))
            
            elif label == 'broken_end':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    broken_end = int(box.get('frame'))
        
        return broken_start, broken_end
    
    except Exception as e:
        print(f"❌ XML 파싱 오류 ({xml_file}): {e}")
        return None, None

def extract_broken_frames_fixed(video_dir, xml_dir, output_dir):
    """수정된 broken 구간 프레임 추출"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 비디오 파일 목록
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    
    print(f"총 {len(video_files)}개 비디오에서 broken 구간 추출 시작...")
    
    total_broken_frames = 0
    videos_with_broken = 0
    videos_without_broken = 0
    
    for video_file in tqdm(video_files, desc="비디오 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # 대응하는 XML 파일 찾기
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        if not os.path.exists(xml_path):
            print(f"⚠️ XML 파일 없음: {xml_file}")
            continue
        
        # broken 정보 추출 (수정된 함수)
        broken_start, broken_end = parse_broken_info_fixed(xml_path)
        
        if broken_start is None or broken_end is None:
            videos_without_broken += 1
            continue
        
        # 비디오 열기
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"❌ 비디오 열기 실패: {video_file}")
            continue
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"\n📹 {video_file}")
        print(f"   broken 구간: {broken_start} ~ {broken_end}")
        
        # broken 구간 유효성 검사
        if broken_end >= total_frames:
            print(f"   ⚠️ broken_end({broken_end})가 총 프레임({total_frames})보다 큼")
            broken_end = total_frames - 1
        
        frame_count = 0
        saved_frames = 0
        
        # 프레임별로 읽기
        while frame_count < total_frames:
            ret, frame = cap.read()
            
            if not ret:
                break
            
            # broken 구간에 있는 프레임만 저장
            if broken_start <= frame_count <= broken_end:
                # 파일명: 비디오이름_프레임번호_broken.jpg
                output_filename = f"{video_name}_{frame_count:03d}_broken.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                # JPG로 저장
                if cv2.imwrite(output_path, frame):
                    saved_frames += 1
            
            frame_count += 1
        
        cap.release()
        
        if saved_frames > 0:
            videos_with_broken += 1
            total_broken_frames += saved_frames
            print(f"   ✅ {saved_frames}개 broken 프레임 저장")
    
    print(f"\n🎉 추출 완료!")
    print(f"   broken 있는 비디오: {videos_with_broken}개")
    print(f"   broken 없는 비디오: {videos_without_broken}개") 
    print(f"   총 broken 프레임: {total_broken_frames}개")
    
    return total_broken_frames

def extract_normal_frames_fixed(video_dir, xml_dir, output_dir, sampling_rate=10):
    """
    정상 구간(broken가 아닌 구간) 프레임을 샘플링해서 저장 (수정된 버전)
    sampling_rate: N프레임마다 1개씩 저장 (기본값: 10프레임마다 1개)
    파일명: 비디오이름_프레임번호_normal.jpg
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    total_normal_frames = 0
    
    print(f"총 {len(video_files)}개 비디오에서 정상 구간 추출 시작... ({sampling_rate}프레임당 1개)")
    
    for video_file in tqdm(video_files, desc="정상 구간 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # XML에서 broken 구간 정보
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        # 수정된 파싱 함수 사용
        broken_start,broken_end = parse_broken_info_fixed(xml_path) if os.path.exists(xml_path) else (None, None)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
            
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_count = 0
        saved_normal = 0
        
        while frame_count < total_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 정상 구간 판단 (broken 구간이 아닌 곳)
            is_normal = True
            if broken_start is not None and broken_end is not None:
                if broken_start <= frame_count <= broken_end:
                    is_normal = False
            
            # 정상 구간이고 샘플링 조건에 맞으면 저장
            if is_normal and frame_count % sampling_rate == 0:
                output_filename = f"{video_name}_{frame_count:03d}_normal.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                if cv2.imwrite(output_path, frame):
                    saved_normal += 1
            
            frame_count += 1
        
        cap.release()
        total_normal_frames += saved_normal
        
        if saved_normal > 0 and broken_start is not None:
            print(f"   ✅ {video_file}: {saved_normal}개 정상 프레임")
    
    print(f"\n🎉 정상 구간 추출 완료!")
    print(f"   총 정상 프레임: {total_normal_frames}개")
    
    return total_normal_frames

# =============================================================================
# 완전한 실행 코드 (broken + normal)
# =============================================================================

# 경로 설정
video_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/videos/breaking_behavior/val/video"
xml_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/videos/breaking_behavior/val/label"
broken_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/frames/broken/val/images"
normal_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/frames/broken/normal/val/images"

print("🚀 절도 감지용 데이터셋 구성 시작!")
print("="*50)

# 1단계: broken 구간 프레임 추출 (수정된 함수)
print("\n📍 1단계: broken 구간 추출")
broken_frames = extract_broken_frames_fixed(video_dir, xml_dir, broken_output_dir)

# 2단계: 정상 구간 프레임 추출 (수정된 함수)
print("\n📍 2단계: 정상 구간 추출")
normal_frames = extract_normal_frames_fixed(video_dir, xml_dir, normal_output_dir, sampling_rate=10)

# 최종 결과
print("\n" + "="*50)
print("🎉 데이터셋 구성 완료!")
print(f"📊 최종 결과:")
print(f"   🔴 brokenn 프레임 : {broken_frames:,}개")
print(f"   🟢 Normal 프레임: {normal_frames:,}개")
print(f"   📈 총 프레임: {broken_frames + normal_frames:,}개")
if broken_frames > 0:
    print(f"   ⚖️ 비율 (normal:broken): {normal_frames // broken_frames}:1")
print("="*50)

🚀 절도 감지용 데이터셋 구성 시작!

📍 1단계: broken 구간 추출
총 80개 비디오에서 broken 구간 추출 시작...


비디오 처리:   0%|          | 0/80 [00:00<?, ?it/s]


📹 C_3_8_59_BU_DYB_10-17_13-22-05_CD_RGB_DF2_M3.mp4
   broken 구간: 66 ~ 127


비디오 처리:   1%|▏         | 1/80 [00:05<07:29,  5.69s/it]

   ✅ 62개 broken 프레임 저장

📹 C_3_8_61_BU_DYB_10-17_13-26-05_CE_RGB_DF2_F3.mp4
   broken 구간: 80 ~ 137


...

   ✅ C_3_8_56_BU_SMC_10-14_13-39-12_CA_RGB_DF2_F3.mp4: 10개 정상 프레임


정상 구간 처리:  99%|█████████▉| 79/80 [04:59<00:03,  3.89s/it]

   ✅ C_3_8_56_BU_DYB_10-17_13-16-04_CD_RGB_DF2_M3.mp4: 12개 정상 프레임


정상 구간 처리: 100%|██████████| 80/80 [05:02<00:00,  3.79s/it]

   ✅ C_3_8_57_BU_DYA_08-23_15-06-11_CD_RGB_DF2_M3.mp4: 10개 정상 프레임

🎉 정상 구간 추출 완료!
   총 정상 프레임: 941개

🎉 데이터셋 구성 완료!
📊 최종 결과:
   🔴 brokenn 프레임 : 5,114개
   🟢 Normal 프레임: 941개
   📈 총 프레임: 6,055개
   ⚖️ 비율 (normal:broken): 0:1
